[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_10/notebook_10_4_counterfactual_explanations.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 10.4: Counterfactual Explanations - "What If" Clinical Scenarios

**Chapter 10: Interpretability and Explainability in Healthcare AI**

## Introduction: Actionable Explanations

A clinician reviews an AI's prediction of diabetes complication risk (the chapter's example patient):

```
Patient: A1C 9.2%, BMI 34, blood pressure 145/92 mmHg
Risk of diabetes complications: HIGH
```

SHAP can say *why* the risk is high ("driven mostly by A1C, then blood pressure, BMI and age").
But the clinician and patient want to know: *"What can we change to lower this risk?"*

**Counterfactual explanations** answer this question: **"What is the smallest realistic change that would move the prediction to a lower risk level?"**

### Why Counterfactuals for Healthcare?

1. **Actionability**: Focus on modifiable risk factors
2. **Personalization**: Tailored to each patient's specific profile
3. **Motivation**: Shows patients concrete targets for improvement
4. **Clinical decision support**: Guides intervention strategies

### Example (from the chapter)

```
Current patient:
  A1C: 9.2%
  Blood pressure: 145/92 mmHg
  → Prediction: HIGH RISK

Counterfactual:
  A1C: 7.0%          ← reduce by 2.2 points (medication adjustment, lifestyle)
  Blood pressure: 130/85  ← reduce by 15/7 mmHg
  → Prediction: MODERATE RISK
```

Some features must **never** be changed by the search: age, diabetes duration, sex and **race**.
We test this example on a synthetic model below, and search for changes that reach **LOW** risk.

---

## Part 1: Setup

In [1]:
# ── Setup (JupyterLite) ─────────────────────────────────────────────────────
import sys
if sys.platform == "emscripten":          # JupyterLite / Pyodide
    import micropip
    await micropip.install(["seaborn"])

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✓ Libraries imported")
print("\nThis notebook demonstrates counterfactual explanations for clinical AI.\n")

✓ Libraries imported

This notebook demonstrates counterfactual explanations for clinical AI.



In [3]:
# Create a SYNTHETIC diabetes complication risk dataset
def create_diabetes_dataset(n_samples=3000):
    """
    Synthetic adults with type 2 diabetes; outcome = diabetes complication within 5 years.
    Risk is driven by A1C, blood pressure, BMI, LDL, smoking, exercise, age and diabetes duration.
    `race` (anonymised categories 0-3) has NO direct effect on the outcome in this generator.
    It only shifts A1C slightly, mimicking unequal access to care: a proxy, not a cause.
    """
    rs = np.random
    age = np.clip(rs.normal(60, 11, n_samples), 30, 85)
    sex = rs.randint(0, 2, n_samples)                       # 1 = male
    race = rs.choice(4, n_samples, p=[0.40, 0.25, 0.20, 0.15])
    duration = np.clip(rs.gamma(2.5, 4, n_samples), 0.5, 40)
    a1c = np.clip(rs.normal(7.8, 1.4, n_samples) + np.array([0, 0.4, 0.3, 0.1])[race], 5.5, 14)
    sbp = np.clip(rs.normal(135, 16, n_samples), 95, 200)
    dbp = np.clip(0.45 * sbp + rs.normal(20, 6, n_samples), 55, 120)
    bmi = np.clip(rs.normal(31, 5.5, n_samples), 18, 50)
    ldl = np.clip(rs.normal(105, 30, n_samples), 40, 250)
    smoking = (rs.random(n_samples) < 0.18).astype(int)
    exercise = np.clip(rs.gamma(1.5, 1.2, n_samples), 0, 10)

    logit = (-1.15 + 0.95 * (a1c - 7.5) + 0.03 * (sbp - 135) + 0.03 * (dbp - 82) + 0.15 * (bmi - 31)
             + 0.05 * (age - 60) + 0.06 * (duration - 10) + 0.006 * (ldl - 105)
             + 0.6 * smoking - 0.2 * (exercise - 1.8))
    complication = (rs.random(n_samples) < 1 / (1 + np.exp(-logit))).astype(int)

    return pd.DataFrame({
        'age': age, 'sex': sex, 'race': race, 'diabetes_duration': duration,
        'a1c': a1c, 'systolic_bp': sbp, 'diastolic_bp': dbp, 'bmi': bmi, 'ldl': ldl,
        'smoking': smoking, 'exercise_hours': exercise, 'complication': complication
    })

# Create dataset
df = create_diabetes_dataset(n_samples=3000)

print("Diabetes Complication Risk Dataset Created (synthetic)")
print(f"\nSamples: {len(df):,}")
print(f"Complication rate: {df['complication'].mean():.1%}")
print("\nFeatures:")
for col in df.columns:
    print(f"  {col}")
df.head(10)


Diabetes Complication Risk Dataset Created (synthetic)

Samples: 3,000
Complication rate: 41.7%

Features:
  age
  sex
  race
  diabetes_duration
  a1c
  systolic_bp
  diastolic_bp
  bmi
  ldl
  smoking
  exercise_hours
  complication


,age,sex,race,diabetes_duration,a1c,systolic_bp,diastolic_bp,bmi,ldl,smoking,exercise_hours,complication
0,65.463856,1,2,4.186802,8.731887,130.137238,70.709087,26.857221,114.341805,0,3.004456,0
1,58.479093,1,3,13.443889,6.246329,120.445568,80.110206,28.575646,139.941991,0,4.654747,0
2,67.124574,1,1,5.966901,8.098049,154.794416,84.957634,39.033061,125.063924,1,1.535325,1
3,76.753328,1,1,3.930145,9.373220,155.995328,94.909430,37.390497,111.301908,0,0.747672,1
4,57.424313,0,1,1.904782,8.059026,120.854254,67.488587,32.146515,111.169640,1,0.615223,0
5,57.424493,0,0,7.663064,6.475161,132.539437,85.831242,18.698452,121.146879,0,0.405178,0
6,77.371341,0,3,2.296798,9.544740,132.705229,81.648364,44.811229,84.858794,0,2.530299,0
7,68.441782,1,3,4.687913,5.500000,134.477506,75.826731,30.890376,83.586016,0,0.328765,0
8,54.835782,0,0,8.571498,7.341087,136.028718,85.361061,29.280881,147.762178,0,0.137208,0
9,65.968160,1,1,17.418088,8.141676,150.149783,84.025232,28.127020,160.932410,0,0.396558,1


In [4]:
# Prepare data
feature_cols = [col for col in df.columns if col != 'complication']
X = df[feature_cols].values
y = df['complication'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {len(X_train):,} | Test: {len(X_test):,}")

Training: 2,400 | Test: 600


## Part 2: Train Model

In [5]:
# Train Random Forest
print("Training diabetes complication risk model...\n")

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=1  # single-threaded: the counterfactual search calls predict_proba on one row thousands of times,
              # and n_jobs=-1 would spawn workers (and a warning) on every call
)

model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]

# Risk bands used throughout the notebook
LOW_RISK, HIGH_RISK = 0.20, 0.50
def risk_band(p):
    return 'LOW' if p < LOW_RISK else ('MODERATE' if p < HIGH_RISK else 'HIGH')

print(f"✓ Model trained")
print(f"\nTest AUC: {roc_auc_score(y_test, y_prob):.3f}")
print(f"Risk bands: LOW < {LOW_RISK:.0%} ≤ MODERATE < {HIGH_RISK:.0%} ≤ HIGH")
print("Test patients per band: " + ", ".join(
    f"{b} {np.mean([risk_band(p) == b for p in y_prob]):.0%}" for b in ['LOW', 'MODERATE', 'HIGH']) + "\n")


Training diabetes complication risk model...



✓ Model trained

Test AUC: 0.860
Risk bands: LOW < 20% ≤ MODERATE < 50% ≤ HIGH
Test patients per band: LOW 22%, MODERATE 42%, HIGH 36%



## Part 3: Counterfactual Generation

### Define Constraints

Not all changes are valid in healthcare (the chapter's three kinds of constraint):
- **Immutable**: age, sex, diabetes duration, **race**: the search never touches them
- **Directional**: A1C, blood pressure, BMI and LDL may only decrease; smoking may only stop; exercise may only increase
- **Range**: each feature stays within realistic limits, and a single plan may change it by at most a realistic amount
  (e.g. A1C by ≤ 3 points, BMI by ≤ 5 units)

### Why `race` must never be an actionable lever

Race is not something a patient can or should change, so a counterfactual that "changes race" is meaningless as advice.
Worse, if changing race *would* move the prediction, that tells us the model uses race as a **proxy** for something else
(here, unequal A1C from unequal access to care), which is a fairness problem to fix in the model and the care pathway,
not a recommendation to give a patient. We keep `race` in the model only so we can *show* this with SHAP in Part 8.

In [6]:
class CounterfactualGenerator:
    """
    Generate counterfactual explanations for healthcare AI.

    Uses a greedy step search, which works for any model, including tree ensembles.
    (A gradient-style search fails on random forests: their predictions are piecewise
    constant, so tiny nudges usually change nothing and the numerical gradient is zero.)
    """
    def __init__(self, model, feature_names, feature_ranges, mutable_features, directions=None, max_change=None):
        self.model = model
        self.feature_names = feature_names
        self.feature_ranges = feature_ranges
        self.mutable_features = mutable_features
        # directions: {feature_index: -1 or +1} restricts a feature to only decrease / increase
        self.directions = directions or {}
        # max_change: {feature_index: largest realistic change in one plan}
        self.max_change = max_change or {}

    def generate(self, x_original, target_class, max_iterations=200, learning_rate=0.05, threshold=0.5):
        """
        Greedy counterfactual search.

        Each iteration tries moving every mutable feature one step up or down
        (step = learning_rate x the feature's allowed range; binary features are toggled)
        and keeps the single move that brings the prediction closest to the target class.
        It stops when the prediction crosses `threshold` (e.g. below the LOW-risk cut-off),
        or when no allowed move helps (local optimum).

        Parameters:
        - x_original: Original patient features
        - target_class: Desired prediction (0 or 1)
        - max_iterations: Maximum number of steps
        - learning_rate: Step size as a fraction of each feature's range
        - threshold: probability the prediction must cross (default 0.5)

        Returns:
        - x_counterfactual: Modified features
        - changes: Dictionary of feature changes
        """
        x_cf = np.array(x_original, dtype=float).copy()
        lo = np.array([r[0] for r in self.feature_ranges], dtype=float)
        hi = np.array([r[1] for r in self.feature_ranges], dtype=float)
        is_binary = (lo == 0) & (hi == 1)
        steps = np.where(is_binary, 1.0, learning_rate * (hi - lo))
        better = -1 if target_class == 0 else 1   # direction in which the probability should move
        # realistic bounds per feature: the allowed range, narrowed by the maximum change per plan
        x0 = np.array(x_original, dtype=float)
        mc = np.array([self.max_change.get(i, np.inf) for i in range(len(x0))])
        lo_i = np.minimum(np.maximum(lo, x0 - mc), x0)
        hi_i = np.maximum(np.minimum(hi, x0 + mc), x0)

        for iteration in range(max_iterations):
            pred_prob = self.model.predict_proba([x_cf])[0, 1]
            if (pred_prob > threshold) == (target_class == 1):
                break  # reached the target class

            candidates = []
            for i, mutable in enumerate(self.mutable_features):
                if not mutable:
                    continue  # never touch immutable features
                for d in (-1, 1):
                    if self.directions.get(i, d) != d:
                        continue  # e.g. BMI may only decrease, smoking may only stop
                    x_try = x_cf.copy()
                    # clip only the feature being moved to its realistic bounds
                    x_try[i] = np.clip(x_try[i] + d * steps[i], lo_i[i], hi_i[i])
                    if x_try[i] != x_cf[i]:
                        candidates.append(x_try)

            if not candidates:
                break
            probs = self.model.predict_proba(np.array(candidates))[:, 1]  # one batched call
            best = int(np.argmax(probs * better))
            if (probs[best] - pred_prob) * better <= 0:
                break  # no allowed move improves the prediction: local optimum
            x_cf = candidates[best]

        # Calculate changes
        changes = {}
        for i, name in enumerate(self.feature_names):
            if abs(x_cf[i] - x_original[i]) > 0.01:
                changes[name] = {
                    'original': x_original[i],
                    'counterfactual': x_cf[i],
                    'change': x_cf[i] - x_original[i]
                }

        return x_cf, changes

print("✓ CounterfactualGenerator class defined\n")


✓ CounterfactualGenerator class defined



In [7]:
# Define constraints (index order follows feature_cols)
print("Feature order:", feature_cols, "\n")
feature_ranges = {
    'age': (30, 85),                # immutable
    'sex': (0, 1),                  # immutable
    'race': (0, 3),                 # immutable, never an actionable lever
    'diabetes_duration': (0.5, 40), # immutable
    'a1c': (6.0, 14.0),             # modifiable, %; not pushed below 6.0 (hypoglycaemia risk)
    'systolic_bp': (110, 200),      # modifiable, mmHg
    'diastolic_bp': (65, 120),      # modifiable, mmHg
    'bmi': (20, 50),                # modifiable
    'ldl': (60, 250),               # modifiable, mg/dL
    'smoking': (0, 1),              # modifiable, binary
    'exercise_hours': (0, 7),       # modifiable, hours/week
}
IMMUTABLE = {'age', 'sex', 'race', 'diabetes_duration'}
feature_ranges_list = [feature_ranges[f] for f in feature_cols]
mutable_features = [f not in IMMUTABLE for f in feature_cols]
idx = {f: i for i, f in enumerate(feature_cols)}

cf_generator = CounterfactualGenerator(
    model=model,
    feature_names=feature_cols,
    feature_ranges=feature_ranges_list,
    mutable_features=mutable_features,
    directions={idx['a1c']: -1, idx['systolic_bp']: -1, idx['diastolic_bp']: -1, idx['bmi']: -1,
                idx['ldl']: -1, idx['smoking']: -1, idx['exercise_hours']: +1},
    max_change={idx['a1c']: 3.0, idx['systolic_bp']: 30, idx['diastolic_bp']: 15, idx['bmi']: 5,
                idx['ldl']: 60, idx['exercise_hours']: 4},
)

print("✓ Counterfactual generator initialized\n")
print("Mutable features:")
for feat, mut in zip(feature_cols, mutable_features):
    status = "✓ Modifiable" if mut else "✗ Immutable"
    print(f"  {feat}: {status}")


Feature order: ['age', 'sex', 'race', 'diabetes_duration', 'a1c', 'systolic_bp', 'diastolic_bp', 'bmi', 'ldl', 'smoking', 'exercise_hours'] 

✓ Counterfactual generator initialized

Mutable features:
  age: ✗ Immutable
  sex: ✗ Immutable
  race: ✗ Immutable
  diabetes_duration: ✗ Immutable
  a1c: ✓ Modifiable
  systolic_bp: ✓ Modifiable
  diastolic_bp: ✓ Modifiable
  bmi: ✓ Modifiable
  ldl: ✓ Modifiable
  smoking: ✓ Modifiable
  exercise_hours: ✓ Modifiable


## Part 4: The Chapter's Example Patient (High Risk)

A1C 9.2%, BMI 34 and blood pressure 145/92 come from the chapter. The remaining values (age 68, male,
10 years since diagnosis, LDL 110, non-smoker, 1 h/week exercise, race category 0) are illustrative choices.

In [8]:
BINARY = {'sex', 'smoking'}
def fmt(feat, val):
    if feat in BINARY:
        return ('Male' if val == 1 else 'Female') if feat == 'sex' else ('Yes' if val == 1 else 'No')
    if feat == 'race':
        return f"category {int(val)}"
    return f"{val:.1f}"

book_patient = {'age': 68, 'sex': 1, 'race': 0, 'diabetes_duration': 10, 'a1c': 9.2, 'systolic_bp': 145,
                'diastolic_bp': 92, 'bmi': 34, 'ldl': 110, 'smoking': 0, 'exercise_hours': 1.0}
patient_original = np.array([book_patient[f] for f in feature_cols], dtype=float)
patient_prob_original = model.predict_proba([patient_original])[0, 1]
patient_idx = 'book_example'

print("="*80)
print("CHAPTER EXAMPLE PATIENT: Counterfactual Analysis")
print("="*80 + "\n")
print(f"Current complication risk: {patient_prob_original:.1%}  → {risk_band(patient_prob_original)} RISK\n")

REF = {'a1c': "(Typical target: <7.0%)", 'systolic_bp': "(Typical target: <130 mmHg)",
       'diastolic_bp': "(Typical target: <80 mmHg)", 'bmi': "(Normal: 18.5-24.9)",
       'ldl': "(Typical target: <100 mg/dL)", 'exercise_hours': "(Recommended: ≥2.5 hrs/week)"}
print("Current Profile:")
for feat_name, feat_val in zip(feature_cols, patient_original):
    print(f"  {feat_name}: {fmt(feat_name, feat_val)} {REF.get(feat_name, '')}")

# The chapter's counterfactual, tested on THIS model (computed, not assumed)
book_cf = patient_original.copy()
book_cf[idx['a1c']], book_cf[idx['systolic_bp']], book_cf[idx['diastolic_bp']] = 7.0, 130, 85
p_book_cf = model.predict_proba([book_cf])[0, 1]
print(f"\nChapter's counterfactual (A1C 9.2→7.0, BP 145/92→130/85): "
      f"{patient_prob_original:.1%} ({risk_band(patient_prob_original)}) → {p_book_cf:.1%} ({risk_band(p_book_cf)})")
if risk_band(patient_prob_original) == 'HIGH' and risk_band(p_book_cf) == 'MODERATE':
    print("✓ Matches the chapter: HIGH → MODERATE. Reaching LOW needs more; the search below looks for it.")
else:
    print("ℹ️ On this synthetic model the result differs from the chapter's HIGH → MODERATE illustration.")
print("\n" + "="*80)


CHAPTER EXAMPLE PATIENT: Counterfactual Analysis

Current complication risk: 88.8%  → HIGH RISK

Current Profile:
  age: 68.0 
  sex: Male 
  race: category 0 
  diabetes_duration: 10.0 
  a1c: 9.2 (Typical target: <7.0%)
  systolic_bp: 145.0 (Typical target: <130 mmHg)
  diastolic_bp: 92.0 (Typical target: <80 mmHg)
  bmi: 34.0 (Normal: 18.5-24.9)
  ldl: 110.0 (Typical target: <100 mg/dL)
  smoking: No 
  exercise_hours: 1.0 (Recommended: ≥2.5 hrs/week)

Chapter's counterfactual (A1C 9.2→7.0, BP 145/92→130/85): 88.8% (HIGH) → 44.7% (MODERATE)
✓ Matches the chapter: HIGH → MODERATE. Reaching LOW needs more; the search below looks for it.



In [9]:
# Generate counterfactual
print(f"\nGenerating counterfactual (finding minimal changes to reach LOW RISK, < {LOW_RISK:.0%})...\n")

patient_cf, changes = cf_generator.generate(
    x_original=patient_original,
    target_class=0,  # Want to flip to low risk
    max_iterations=2000,
    learning_rate=0.05,
    threshold=LOW_RISK
)

# Predict on counterfactual
patient_prob_cf = model.predict_proba([patient_cf])[0, 1]

print(f"✓ Counterfactual generated\n")
print(f"Counterfactual complication risk: {patient_prob_cf:.1%}  "
      f"{'✓ LOW RISK' if patient_prob_cf < LOW_RISK else '⚠️ Not low: ' + risk_band(patient_prob_cf)}\n")

found_cf = patient_prob_cf < LOW_RISK   # a valid counterfactual must actually reach low risk

if len(changes) == 0 and patient_prob_original < LOW_RISK:
    print("✓ No changes needed: patient is already low risk.\n")
elif not found_cf:
    print("⚠️ NO VALID COUNTERFACTUAL FOUND")
    print(f"   Risk after the search: {patient_prob_original:.1%} → {patient_prob_cf:.1%} (target: below {LOW_RISK:.0%})")
    if changes:
        print("   Partial changes tried (NOT sufficient, do not present as a plan): " + ", ".join(changes))
    print("\n   Why? Either the greedy search reached a local optimum (no single allowed step lowers")
    print("   risk further), or this patient's risk is driven mainly by IMMUTABLE factors (age,")
    print("   diabetes duration) that no realistic allowed change can offset.")
    print("\n   Exercise: which immutable features drive this patient's risk? Try a larger step size.\n")
else:
    print("="*80)
    print("COUNTERFACTUAL EXPLANATION (What needs to change?)")
    print("="*80 + "\n")

    print(f"Risk reduction: {patient_prob_original:.1%} → {patient_prob_cf:.1%} ({patient_prob_original - patient_prob_cf:.1%} decrease)\n")

    print("Required Changes:\n")

    for i, (feat_name, change_info) in enumerate(changes.items(), 1):
        orig = change_info['original']
        cf = change_info['counterfactual']
        delta = change_info['change']

        # Format based on feature type
        if feat_name in BINARY:
            orig_str, cf_str = fmt(feat_name, orig), fmt(feat_name, cf)
            delta_str = f"Change to {cf_str}"
        else:
            orig_str = f"{orig:.1f}"
            cf_str = f"{cf:.1f}"
            delta_str = f"{delta:+.1f}"

        print(f"{i}. {feat_name.replace('_', ' ').title()}")
        print(f"   Current: {orig_str}")
        print(f"   Target: {cf_str}  ({delta_str})")

        # Clinical recommendations
        if feat_name == 'a1c' and delta < 0:
            print(f"   💊 Intervention: Diabetes medication review + diet/activity (glycaemic control)")
        elif feat_name in ('systolic_bp', 'diastolic_bp') and delta < 0:
            print(f"   💊 Intervention: Blood pressure control (e.g., ACE inhibitor/ARB, less salt)")
        elif feat_name == 'ldl' and delta < 0:
            print(f"   💊 Intervention: Lipid lowering (e.g., statin therapy)")
        elif feat_name == 'bmi' and delta < 0:
            print(f"   🏃 Intervention: Weight loss program (diet + exercise)")
        elif feat_name == 'smoking' and orig == 1 and cf == 0:
            print(f"   🚭 Intervention: Smoking cessation (nicotine replacement, counseling)")
        elif feat_name == 'exercise_hours' and delta > 0:
            print(f"   🏃 Intervention: Increase aerobic exercise to {cf:.0f} hrs/week")

        print()

    print("="*80)

    # Clinical summary
    print("\n💡 CLINICAL ACTION PLAN:\n")
    print("If this patient achieves the above targets, their predicted complication risk")
    print(f"would decrease from {patient_prob_original:.1%} to {patient_prob_cf:.1%}.\n")

    print("Recommended follow-up:")
    print("  • Initiate pharmacotherapy as indicated")
    print("  • Referral to nutrition/lifestyle counseling")
    print("  • Recheck A1C in 3 months")
    print("  • Reassess complication risk after intervention\n")


Generating counterfactual (finding minimal changes to reach LOW RISK, < 20%)...



✓ Counterfactual generated

Counterfactual complication risk: 19.8%  ✓ LOW RISK

COUNTERFACTUAL EXPLANATION (What needs to change?)

Risk reduction: 88.8% → 19.8% (69.0% decrease)

Required Changes:

1. A1C
   Current: 9.2
   Target: 6.4  (-2.8)
   💊 Intervention: Diabetes medication review + diet/activity (glycaemic control)

2. Bmi
   Current: 34.0
   Target: 29.0  (-5.0)
   🏃 Intervention: Weight loss program (diet + exercise)

3. Ldl
   Current: 110.0
   Target: 100.5  (-9.5)
   💊 Intervention: Lipid lowering (e.g., statin therapy)


💡 CLINICAL ACTION PLAN:

If this patient achieves the above targets, their predicted complication risk
would decrease from 88.8% to 19.8%.

Recommended follow-up:
  • Initiate pharmacotherapy as indicated
  • Referral to nutrition/lifestyle counseling
  • Recheck A1C in 3 months
  • Reassess complication risk after intervention



### Visualize Counterfactual Changes

In [10]:
if len(changes) > 0 and found_cf:   # only plot a counterfactual that actually reaches low risk
    # Create comparison plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Plot 1: Feature comparison
    ax1 = axes[0]

    changed_features = list(changes.keys())
    orig_values = [changes[f]['original'] for f in changed_features]
    cf_values = [changes[f]['counterfactual'] for f in changed_features]

    x = np.arange(len(changed_features))
    width = 0.35

    bars1 = ax1.barh(x - width/2, orig_values, width, label='Current',
                    color='red', alpha=0.7, edgecolor='black')
    bars2 = ax1.barh(x + width/2, cf_values, width, label='Target (Counterfactual)',
                    color='green', alpha=0.7, edgecolor='black')

    ax1.set_yticks(x)
    ax1.set_yticklabels([f.replace('_', ' ').title() for f in changed_features])
    ax1.set_xlabel('Feature Value', fontsize=12, fontweight='bold')
    ax1.set_title('Current vs Target Profile', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='x')

    # Plot 2: Risk comparison
    ax2 = axes[1]

    risks = [patient_prob_original, patient_prob_cf]
    labels = ['Current\nRisk', 'Counterfactual\nRisk']
    colors = ['red', 'green']

    bars = ax2.bar(labels, risks, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

    # Add risk threshold line
    ax2.axhline(y=HIGH_RISK, color='orange', linestyle='--', linewidth=2, label='High-risk threshold')
    ax2.axhline(y=LOW_RISK, color='green', linestyle=':', linewidth=2, label='Low-risk threshold')

    ax2.set_ylabel('Predicted complication risk', fontsize=12, fontweight='bold')
    ax2.set_title('Risk Reduction with Interventions', fontsize=14, fontweight='bold')
    ax2.set_ylim(0, 1.0)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')

    # Add value labels
    for bar, val in zip(bars, risks):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{val:.1%}', ha='center', va='bottom', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print("\n📊 Visualization shows the path from high risk to low risk.\n")
else:
    print("\n⚠️ No visualization (no counterfactual reached low risk)\n")


📊 Visualization shows the path from high risk to low risk.



## Part 5: Multiple Counterfactuals

Sometimes there are multiple ways to achieve the target. Let's explore alternatives.

In [11]:
# Generate multiple counterfactuals with different random initializations
print("Generating diverse counterfactual explanations...\n")

counterfactuals = []

for i in range(3):
    # Slightly different optimization
    cf_patient, cf_changes = cf_generator.generate(
        x_original=patient_original,
        target_class=0,
        max_iterations=2000,
        learning_rate=0.03 + i*0.01,  # Vary learning rate
        threshold=LOW_RISK
    )

    cf_prob = model.predict_proba([cf_patient])[0, 1]

    if cf_prob < LOW_RISK:  # Successfully reached low risk
        counterfactuals.append({
            'features': cf_patient,
            'changes': cf_changes,
            'risk': cf_prob,
            'n_changes': len(cf_changes)
        })

print(f"Generated {len(counterfactuals)} valid counterfactuals\n")

if len(counterfactuals) > 0:
    print("="*80)
    print("ALTERNATIVE INTERVENTION STRATEGIES")
    print("="*80 + "\n")

    for i, cf in enumerate(counterfactuals, 1):
        print(f"Option {i}:  Risk: {patient_prob_original:.1%} → {cf['risk']:.1%}  |  {cf['n_changes']} changes\n")

        for feat_name, change_info in cf['changes'].items():
            orig = change_info['original']
            target = change_info['counterfactual']
            delta = change_info['change']

            if feat_name in BINARY:
                print(f"  • {feat_name.replace('_', ' ').title()}: {fmt(feat_name, orig)} → {fmt(feat_name, target)}")
            else:
                print(f"  • {feat_name.replace('_', ' ').title()}: {orig:.1f} → {target:.1f} ({delta:+.1f})")

        print()

    print("="*80)
    print("\n💡 CLINICAL DECISION:")
    print("\nMultiple intervention pathways can achieve risk reduction.")
    print("Choose based on:")
    print("  • Patient preferences and barriers")
    print("  • Feasibility of interventions")
    print("  • Side effect profiles")
    print("  • Cost and insurance coverage")
    print("  • Comorbidities and contraindications\n")
else:
    print("⚠️ Could not generate multiple valid counterfactuals.\n")

Generating diverse counterfactual explanations...



Generated 3 valid counterfactuals

ALTERNATIVE INTERVENTION STRATEGIES

Option 1:  Risk: 88.8% → 18.7%  |  4 changes

  • A1C: 9.2 → 6.3 (-2.9)
  • Systolic Bp: 145.0 → 142.3 (-2.7)
  • Bmi: 34.0 → 29.0 (-5.0)
  • Ldl: 110.0 → 98.6 (-11.4)

Option 2:  Risk: 88.8% → 19.9%  |  3 changes

  • A1C: 9.2 → 6.6 (-2.6)
  • Diastolic Bp: 92.0 → 87.6 (-4.4)
  • Bmi: 34.0 → 29.2 (-4.8)

Option 3:  Risk: 88.8% → 19.8%  |  3 changes

  • A1C: 9.2 → 6.4 (-2.8)
  • Bmi: 34.0 → 29.0 (-5.0)
  • Ldl: 110.0 → 100.5 (-9.5)


💡 CLINICAL DECISION:

Multiple intervention pathways can achieve risk reduction.
Choose based on:
  • Patient preferences and barriers
  • Feasibility of interventions
  • Side effect profiles
  • Cost and insurance coverage
  • Comorbidities and contraindications



## Part 6: Patient Communication

Translate counterfactuals into patient-friendly language.

In [12]:
def generate_patient_report(patient_original, patient_cf, changes, risk_original, risk_cf, feature_names):
    """
    Generate patient-friendly explanation.
    """
    print("\n" + "="*80)
    print("PATIENT EDUCATION: Understanding Your Diabetes Complication Risk")
    print("="*80 + "\n")

    print(f"Your Current Risk:\n")
    print(f"Based on your results, the model estimates your chance of developing a diabetes")
    print(f"complication (eyes, kidneys, nerves, heart) at about {risk_original:.0%}. This is {risk_band(risk_original)} risk.\n")

    if len(changes) == 0:
        print("You are already at low risk. Keep up the good work!\n")
        return

    print(f"Good News: You Can Lower Your Risk!\n")
    print(f"By making these changes, you could reduce your risk to about {risk_cf:.0%}:\n")

    for i, (feat_name, change_info) in enumerate(changes.items(), 1):
        orig = change_info['original']
        target = change_info['counterfactual']

        if feat_name == 'a1c':
            print(f"{i}. Bring your A1C (average blood sugar) from {orig:.1f}% to {target:.1f}%")
            print(f"   How: Review your diabetes medicines with your doctor, adjust diet, stay active")
            print(f"   Why: High blood sugar damages small blood vessels in eyes, kidneys and nerves\n")

        elif feat_name == 'systolic_bp':
            print(f"{i}. Lower your blood pressure (top number) from {orig:.0f} to {target:.0f} mmHg")
            print(f"   How: Take blood pressure medicine, reduce salt, lose weight")
            print(f"   Why: High blood pressure strains your heart, kidneys and blood vessels\n")

        elif feat_name == 'diastolic_bp':
            print(f"{i}. Lower your blood pressure (bottom number) from {orig:.0f} to {target:.0f} mmHg\n")

        elif feat_name == 'bmi':
            print(f"{i}. Lose weight to reach BMI of {target:.1f} (from {orig:.1f})")
            print(f"   How: Eat healthier, exercise more, work with dietitian")
            print(f"   Why: Losing weight improves blood sugar and blood pressure\n")

        elif feat_name == 'ldl':
            print(f"{i}. Lower your LDL ('bad') cholesterol from {orig:.0f} to {target:.0f} mg/dL")
            print(f"   How: Cholesterol medicine (statin) as prescribed, healthier fats\n")

        elif feat_name == 'smoking' and orig == 1:
            print(f"{i}. Quit smoking")
            print(f"   How: Talk to your doctor about nicotine patches, gum, or pills")
            print(f"   Why: Smoking damages blood vessels and multiplies diabetes risks\n")

        elif feat_name == 'exercise_hours':
            print(f"{i}. Exercise {target:.0f} hours per week (currently {orig:.1f} hours)")
            print(f"   How: Walk 30 minutes daily, find activities you enjoy")
            print(f"   Why: Exercise helps your body use insulin better\n")

    print("Remember: These changes take time. Start with small steps.")
    print("Your doctor will help you create a plan that works for you.\n")

    print("="*80)

# Generate patient report only for a counterfactual that actually reaches low risk
if len(changes) > 0 and patient_prob_cf < LOW_RISK:
    generate_patient_report(
        patient_original=patient_original,
        patient_cf=patient_cf,
        changes=changes,
        risk_original=patient_prob_original,
        risk_cf=patient_prob_cf,
        feature_names=feature_cols
    )
else:
    print("No patient report: no counterfactual reached low risk, so there is no")
    print("evidence-based target to communicate. Never present partial changes as a plan.")



PATIENT EDUCATION: Understanding Your Diabetes Complication Risk

Your Current Risk:

Based on your results, the model estimates your chance of developing a diabetes
complication (eyes, kidneys, nerves, heart) at about 89%. This is HIGH risk.

Good News: You Can Lower Your Risk!

By making these changes, you could reduce your risk to about 20%:

1. Bring your A1C (average blood sugar) from 9.2% to 6.4%
   How: Review your diabetes medicines with your doctor, adjust diet, stay active
   Why: High blood sugar damages small blood vessels in eyes, kidneys and nerves

2. Lose weight to reach BMI of 29.0 (from 34.0)
   How: Eat healthier, exercise more, work with dietitian
   Why: Losing weight improves blood sugar and blood pressure

3. Lower your LDL ('bad') cholesterol from 110 to 100 mg/dL
   How: Cholesterol medicine (statin) as prescribed, healthier fats

Remember: These changes take time. Start with small steps.
Your doctor will help you create a plan that works for you.



## Part 7: Patients at Different Risk Levels

Counterfactuals mean different things at different risk levels. We take one LOW, one MODERATE and the HIGH-risk
example patient and run the same constrained search for each (target: LOW risk).

In [13]:
def pick(band):
    """First test patient in a risk band (choose one near the middle of the band)."""
    cand = np.where([risk_band(p) == band for p in y_prob])[0]
    mid = {'LOW': LOW_RISK / 2, 'MODERATE': (LOW_RISK + HIGH_RISK) / 2}[band]
    return cand[np.argmin(np.abs(y_prob[cand] - mid))]

cases = {'LOW (test patient)': X_test[pick('LOW')].astype(float),
         'MODERATE (test patient)': X_test[pick('MODERATE')].astype(float),
         'HIGH (chapter example)': patient_original}
case_results = {}
for label, x in cases.items():
    p0 = model.predict_proba([x])[0, 1]
    x_cf, ch = cf_generator.generate(x, target_class=0, max_iterations=2000, learning_rate=0.05, threshold=LOW_RISK)
    p1 = model.predict_proba([x_cf])[0, 1]
    ok = p1 < LOW_RISK
    case_results[label] = dict(x=x, x_cf=x_cf, changes=ch, p0=p0, p1=p1, ok=ok)
    assert all(abs(x_cf[idx[f]] - x[idx[f]]) < 1e-9 for f in IMMUTABLE), "immutable feature changed!"

    print(f"{label}: risk {p0:.1%} ({risk_band(p0)})")
    if p0 < LOW_RISK:
        print("   Already LOW: no change needed. Reinforce current care.\n")
    elif ok:
        print(f"   ✓ Reaches LOW ({p1:.1%}) with {len(ch)} change(s): " +
              "; ".join(f"{f} {fmt(f, c['original'])}→{fmt(f, c['counterfactual'])}" for f, c in ch.items()) + "\n")
    else:
        print(f"   ✗ No realistic plan reaches LOW (best found {p1:.1%}, {risk_band(p1)}).")
        print("     Do not present this as a plan; the remaining risk is driven by factors outside the allowed changes.\n")

print("✓ Immutable features (age, sex, race, diabetes duration) were never changed in any search.")


LOW (test patient): risk 10.0% (LOW)
   Already LOW: no change needed. Reinforce current care.

MODERATE (test patient): risk 35.0% (MODERATE)
   ✓ Reaches LOW (19.8%) with 1 change(s): bmi 32.5→29.5

HIGH (chapter example): risk 88.8% (HIGH)
   ✓ Reaches LOW (19.8%) with 3 change(s): a1c 9.2→6.4; bmi 34.0→29.0; ldl 110.0→100.5

✓ Immutable features (age, sex, race, diabetes duration) were never changed in any search.


## Part 8: Counterfactuals vs SHAP for the Same Patients

SHAP explains *why* the model predicts what it does; a counterfactual says *what to change*.
Do they point at the same features? We compute TreeSHAP for the same patients and compare:
- **SHAP share on immutable features**: how much of the explanation the patient cannot act on (including race)
- **Agreement**: is the mutable feature with the largest risk-increasing SHAP value among those the counterfactual changed?

*TreeSHAP needs the `shap` package, which is not available in the browser (JupyterLite) version. This part is
skipped there; run it in Colab or local Jupyter.*

In [14]:
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("ℹ️ This section needs SHAP (Colab or local Jupyter); it is skipped in the browser version.")

if HAS_SHAP:
    from scipy.stats import spearmanr
    explainer = shap.TreeExplainer(model)
    def shap_pos(x):
        sv = explainer.shap_values(np.atleast_2d(x))
        if isinstance(sv, list):          # older SHAP: [class_0, class_1]
            sv = sv[1]
        sv = np.asarray(sv)
        if sv.ndim == 3:                  # SHAP >= 0.45: (samples, features, classes)
            sv = sv[:, :, 1]
        return sv[0]

    mutable_names = [f for f in feature_cols if f not in IMMUTABLE]
    agree, compared, race_shares = 0, 0, []
    for label, r in case_results.items():
        sv = pd.Series(shap_pos(r['x']), index=feature_cols)
        imm_share = sv[list(IMMUTABLE)].abs().sum() / sv.abs().sum()
        print(f"{label}: risk {r['p0']:.1%}")
        top4 = sv.reindex(sv.abs().sort_values(ascending=False).index[:4])
        print("   Top SHAP drivers: " + ", ".join(f"{f} {v:+.3f}" for f, v in top4.items()))
        print(f"   Share of |SHAP| on immutable features: {imm_share:.0%} (race alone: {abs(sv['race']) / sv.abs().sum():.1%})")
        race_shares.append(abs(sv['race']) / sv.abs().sum())
        if r['p0'] < LOW_RISK:
            print("   Already LOW: no counterfactual needed, nothing to compare\n")
            continue
        if not r['ok']:
            print("   No valid counterfactual reached LOW, nothing to compare\n")
            continue
        pos_mut = sv[mutable_names][sv[mutable_names] > 0].sort_values(ascending=False)
        changed = list(r['changes'])
        top_mut = pos_mut.index[0] if len(pos_mut) else None
        hit = top_mut in changed
        compared += 1; agree += hit
        print(f"   Largest risk-raising MUTABLE feature by SHAP: {top_mut}  |  counterfactual changed: {changed}")
        print(f"   {'✓ Agree' if hit else '✗ Disagree'}: the counterfactual {'does' if hit else 'does NOT'} act on SHAP's top modifiable driver")
        # Rank agreement over mutable features: SHAP value vs size of the change (as a fraction of its allowed range)
        rel_change = pd.Series({f: abs(r['x_cf'][idx[f]] - r['x'][idx[f]]) /
                                (feature_ranges[f][1] - feature_ranges[f][0]) for f in mutable_names})
        rho = spearmanr(sv[mutable_names], rel_change).correlation
        print(f"   Spearman ρ (SHAP value vs relative change, mutable features) = {rho:.2f}")
        not_pos = [f for f in changed if sv[f] <= 0]
        if not_pos:
            print(f"   Note: changed features with SHAP ≤ 0 for this patient: {not_pos} (a change can help even when")
            print("   the current value is not pushing risk up: SHAP is relative to the average patient)")
        print()

    if compared:
        print(f"SHAP's top modifiable driver was acted on in {agree}/{compared} counterfactual(s).")
    # The chapter's illustrative SHAP split for its example patient, compared with this model (computed)
    sv_book = pd.Series(np.abs(shap_pos(patient_original)), index=feature_cols)
    groups = {'A1C': ['a1c'], 'blood pressure': ['systolic_bp', 'diastolic_bp'], 'BMI': ['bmi'], 'age': ['age']}
    BOOK_SPLIT = {'A1C': 0.40, 'blood pressure': 0.25, 'BMI': 0.20, 'age': 0.15}
    print("\nChapter example patient: share of |SHAP| by factor (this model vs the chapter's illustration)")
    for g, fs in groups.items():
        print(f"  {g:<16}{sv_book[fs].sum() / sv_book.sum():>5.0%}   (chapter: {BOOK_SPLIT[g]:.0%})")
    other = 1 - sum(sv_book[fs].sum() for fs in groups.values()) / sv_book.sum()
    print(f"  {'other features':<16}{other:>5.0%}   (the chapter lists only these four factors)")
    ours = sorted(groups, key=lambda g: -sv_book[groups[g]].sum())
    book_order = sorted(BOOK_SPLIT, key=lambda g: -BOOK_SPLIT[g])
    if ours == book_order:
        print("  ✓ Same ranking as the chapter's illustration (exact percentages vary between models and samples).")
    else:
        print(f"  Ranking here: {' > '.join(ours)}; chapter: {' > '.join(book_order)}.")
        print("  The chapter's split is illustrative; a different model or sample shifts these shares.")
    print("\n💡 SHAP and counterfactuals answer different questions. SHAP spreads credit across ALL features,")
    print("   including age, diabetes duration and race, which no one can change. The counterfactual is")
    print("   restricted to actionable features, so it can be shorter, and it may favour a feature that is")
    print("   cheap to move under the step sizes even if SHAP ranks it lower.")
    print(f"\n   Race carries at most {max(race_shares):.1%} of |SHAP| for these patients.")
    if max(race_shares) >= 0.05:
        print("   ⚠️ That is noticeable: the model uses race as a proxy. Investigate (Chapter 9); never use it as a lever.")
    else:
        print("   Small here, because A1C (the variable race was a proxy for) is measured directly. With unmeasured")
        print("   factors in real data, race can carry real credit: a fairness red flag (Chapter 9), never a lever.")


LOW (test patient): risk 10.0%
   Top SHAP drivers: a1c -0.223, diastolic_bp -0.046, systolic_bp -0.046, age -0.042
   Share of |SHAP| on immutable features: 15% (race alone: 0.0%)
   Already LOW: no counterfactual needed, nothing to compare

MODERATE (test patient): risk 35.0%
   Top SHAP drivers: a1c -0.161, bmi +0.062, diabetes_duration -0.047, age +0.046
   Share of |SHAP| on immutable features: 25% (race alone: 0.3%)
   Largest risk-raising MUTABLE feature by SHAP: bmi  |  counterfactual changed: ['bmi']
   ✓ Agree: the counterfactual does act on SHAP's top modifiable driver
   Spearman ρ (SHAP value vs relative change, mutable features) = 0.61

HIGH (chapter example): risk 88.8%
   Top SHAP drivers: a1c +0.178, bmi +0.127, diastolic_bp +0.067, age +0.044
   Share of |SHAP| on immutable features: 11% (race alone: 0.3%)
   Largest risk-raising MUTABLE feature by SHAP: a1c  |  counterfactual changed: ['a1c', 'bmi', 'ldl']
   ✓ Agree: the counterfactual does act on SHAP's top modifia

## Summary and Key Takeaways

### What We Learned

1. **Counterfactual Explanations**:
   - Answer: "What needs to change to flip the prediction?"
   - Focus on modifiable risk factors (actionable)
   - Provide concrete intervention targets

2. **Clinical Value**:
   - **Actionability**: Guides treatment decisions
   - **Personalization**: Tailored to each patient's profile
   - **Motivation**: Shows patients achievable goals
   - **Shared decision-making**: Patient + physician collaboration

3. **Implementation Challenges**:
   - **Constraints**: Must respect immutability (age, race), monotonicity (weight), plausibility
   - **Multiple solutions**: Often many ways to achieve target
   - **Optimization**: Computationally intensive for complex models

4. **Practical Workflow**:
   ```
   1. Patient presents → AI predicts HIGH RISK
   2. Generate counterfactual: what changes would lower risk?
   3. Clinician reviews: are changes feasible for this patient?
   4. Discuss with patient: which interventions are acceptable?
   5. Create action plan based on counterfactual targets
   6. Monitor progress, reassess risk
   ```

### Best Practices

✅ **Respect immutability**: Don't suggest changing age, race, genetics  
✅ **Prioritize feasibility**: Realistic, achievable changes  
✅ **Generate diverse options**: Multiple pathways to target  
✅ **Communicate clearly**: Translate to patient-friendly language  
✅ **Validate clinically**: Ensure interventions are medically appropriate  

### Limitations

⚠️ **Correlation ≠ Causation**: Changing features may not causally change outcome  
⚠️ **Model assumptions**: Assumes AI model is correct (garbage in, garbage out)  
⚠️ **Ignores interactions**: May not capture complex feature dependencies  
⚠️ **Computational cost**: Optimization can be slow for large models  
⚠️ **Unrealistic changes**: May suggest infeasible combinations  

### Comparison: Counterfactuals vs Other Methods

| Method | Question Answered | Actionability | Intuition |
|--------|-------------------|---------------|----------|
| **SHAP** | Which features drive *this* prediction? | Low (includes immutable) | Moderate |
| **LIME** | What local pattern explains *this* prediction? | Low (includes immutable) | High |
| **Counterfactuals** | What *changes* would flip prediction? | **High** (only modifiable) | **Very High** |

**Key Difference**: Counterfactuals are forward-looking (what to do) vs explanatory (why prediction was made).

### Clinical Impact

**Open questions to test (not established findings)**:
- Hypothesis: concrete counterfactual targets improve patients' adherence to treatment plans; this needs prospective evaluation
- Hypothesis: counterfactuals support shared decision-making (as the chapter argues); measure it with clinicians and patients
- Likely most useful in chronic disease management (diabetes, CVD, obesity), where modifiable factors drive risk

---

**Chapter 10 Complete!** We've covered:
- **10.1**: SHAP (global + local importance)
- **10.2**: LIME (local linear approximations)
- **10.3**: GradCAM (visual explanations for imaging)
- **10.4**: Counterfactuals (actionable "what if" scenarios)

Together, these methods make black box AI transparent, trustworthy, and clinically actionable.

*"The best explanation is one a clinician can act on and a patient can understand."*